# Week 12 作業 — Dataset Hygiene Report

**課程**：NS5116 電腦硬體與程式語言在行為科學實驗與大數據分析之應用  
**資料集**：`./data/messy_stroop_homework.csv`（`seed=2026`，n ≈ 240 trials）

---
## A.0 Data Source & Schema Investigation

本節以**三個獨立來源交叉驗證（triangulation）**的方式調查各欄位定義與合理範圍。
每條清理規則均須標明依據來源，不接受無根據的「感覺合理」。

### 來源 1 — 生成器原始碼 (Generator)

#### 生成器逆向工程紀錄

以下按「欄位定義 → Stroop effect 注入 → 問題注入」的程式執行順序整理。

**Seed 與可重現性**

第 30 行固定 `np.random.seed(2026)`。NumPy 的偽隨機數生成器在 seed 固定時，序列完全確定，因此**同一 NumPy 版本下執行兩次生成器會得到 bit-for-bit 相同的 CSV**。換句話說，這份資料沒有「版本問題」，可重現性有保障；唯一例外是不同 NumPy 主版本可能改變 PRNG 演算法。

**各欄位生成邏輯（第 37–46 行）**

- `trial_id`：`np.arange(1, 241)` → 整數 1–240，後因注入重複列增至 243 列。
- `subject_id`：`np.random.choice([101, 102, 103, 104, 105, 106], 240)` → 6 個 distinct 整數，無 missing 設計。
- `condition`：`np.random.choice(["congruent", "congruent ", "incongruent", "Incongruent"], 240)` → **生成時就已含 4 個變體**，尾端空白與大小寫不一致是第一層 dirty。
- `rt_ms`：`np.random.normal(520, 90, 240)` → 分佈中心 **520 ms、SD 90**。
- `accuracy`：`np.random.choice([0, 1, 1, 1, 1], 240)` → 約 80% 為 1。
- `age`：從 `[22, 26, …, 74, -1, 888, np.nan]`（共 17 個值）中隨機抽取，含**三種 missing 表示法**同時並存。

第 51–54 行在問題注入前先加上 +80 ms Stroop effect（congruent vs incongruent），使真實效應量可查驗。

**刻意注入的 Messy Patterns（附關鍵程式行）**

**Pattern ① — condition 額外縮寫（第 89–91 行）**

在基底 4 變體之上，再覆蓋 6 列為縮寫，使欄位共出現 **6 個表面值**：

```python
# 第 89–91 行
abbrev_idx = np.random.choice(N, 6, replace=False)
df.loc[abbrev_idx[:3], "condition"] = "con"      # 3 列：缩寫 congruent
df.loc[abbrev_idx[3:], "condition"] = "incong."  # 3 列：縮寫 incongruent
```

清理需：`strip()` → `lower()` → `replace/map` 縮寫 → 標準化到 2 個 level。

**Pattern ② — rt_ms 轉為 object dtype 並注入雙字串 sentinel（第 59、62–67 行）**

```python
# 第 59 行：float64 → object，欄位 dtype 損壞
rt_col = df["rt_ms"].astype(object)

# 第 62–63 行：第一種字串 sentinel（8 列）
rt_col.iloc[missing_idx] = "missing"

# 第 66–67 行：第二種字串 sentinel（5 列）
rt_col.iloc[dash_idx] = "--"
```

**Pattern ③ — rt_ms 雙數值 sentinel（第 73–84 行）**

```python
# 第 74–76 行：負值 sentinel，代表「無反應」
rt_col.iloc[neg_idx] = -1      # 4 列

# 第 80–81 行：正值 sentinel，代表 timeout（注意是 9999，非 99999）
rt_col.iloc[pos_idx] = 9999   # 4 列

df["rt_ms"] = rt_col           # 第 84 行：寫回 df，dtype 仍為 object
```

兩種字串（13 列）＋兩種數值（8 列）sentinel 互不重疊，共 **21 列** rt_ms 需修補，是本資料集最複雜的欄位。

**Pattern ④ — accuracy 混入字串 "True"/"False"（第 96–101 行）**

```python
# 第 96 行：int → object
acc_col = df["accuracy"].astype(object)

# 第 99–100 行：10 列改為字串，各半
for i, k in enumerate(tf_idx):
    acc_col.iloc[k] = "True" if i % 2 == 0 else "False"
df["accuracy"] = acc_col
```

**Pattern ⑤ — age 三種 missing 同欄並存（第 44–45 行）**

```python
# 第 44–45 行：-1、888、np.nan 同時在同一 choice 陣列
"age": np.random.choice([22, 26, 30, 34, 38, 42, 46, 50, 54,
                          58, 62, 66, 70, 74, -1, 888, np.nan], N)
```

**Pattern ⑥ — 重複列（第 108–110 行）**

```python
# 第 108–110 行：複製 3 列後 concat，240 → 243 列
dup_indices = np.random.choice(N, 3, replace=False)
duplicates  = df.iloc[dup_indices].copy()
df = pd.concat([df, duplicates], ignore_index=True)
# 第 113 行：shuffle，重複列不在結尾，需用 duplicated() 才能找到
df = df.sample(frac=1.0, random_state=2026).reset_index(drop=True)
```

**小結**

生成器共注入 **6 類、涉及 4 個欄位**的 dirty pattern。`rt_ms` 最複雜（dtype 損壞 + 2 種字串 + 2 種數值 sentinel）；`condition` 有 6 個表面變體但邏輯上只有 2 個；`age` 同欄三種 missing 表示；`accuracy` dtype 混用；加上 3 列完整重複 trial。

---
### 來源 2 — 文獻 (Literature)

**引用（APA 格式）**

Ménétré, E., & Laganaro, M. (2023). The temporal dynamics of the Stroop effect from childhood to young and older adulthood. *PLOS ONE*, *18*(3), e0256003. https://doi.org/10.1371/journal.pone.0256003

**從文獻中摘錄**

| 項目 | 文獻數據 | 出處 |
|------|----------|------|
| RT 典型 mean ± SD | 682 ± 157 ms | Section 3.1，年輕成人組（20–30 歲）|
| RT 絕對下限 | 200 ms（低於此視為猜測／預期反應） | Section 2.4 |
| RT SD-based 下限 | mean − 2.5 SD = 289.5 ms（*該*樣本） | Section 2.4 |
| RT SD-based 上限 | mean + 2.5 SD = 1074.5 ms（*該*樣本） | Section 2.4 |
| Stroop effect 量級 | ~36 ms（LMM 估計值） | Section 3.1，年輕成人組 |

**採用範圍與取捨說明**

本報告採用 **200–3000 ms**。

- **下限 200 ms**：直接取自 Ménétré & Laganaro (2023) Section 2.4 的絕對閾值。此閾值基於生理原則（選擇反應時間低於 200 ms 在人類幾乎不可能），與樣本無關，可直接移植到本資料集。相比課堂 demo 的 150 ms，200 ms 有文獻明確支持，依據更強。
- **上限 3000 ms**：文獻的 SD-based 上限 1074.5 ms 是由*那份研究的樣本* mean(682) ± 2.5 × SD(157) 導出。本資料集 RT 均值約 520 ms（生成器第 43 行 `np.random.normal(520, 90)`），直接套用 1074.5 ms 等於把另一樣本的分佈強加在本資料上，這樣會是方法論錯誤。因此固定上限採課堂 demo 的 3000 ms 以保留試次。
- **SD-based trimming 的去處**：文獻 ±2.5 SD 的方法移至 `analyse(outlier_sd=3.0)`，針對本作業使用的這筆資料的實際分佈去調整會比較適當。
- **Stroop effect 對照**：文獻報告 ~36 ms（LMM 估計）；生成器注入 +80 ms（原始 mean 差）。兩者均為有效參照，差異來自估計方法（LMM vs. 原始 mean 差）與樣本設計不同，不構成矛盾。

---
### 來源 3 — 資料本身 (Data-Driven Discovery)

對**未清理**的原始資料執行初步探索，觀察哪些欄位的分佈與來源 1、2 不一致。
程式碼在下一格執行，解讀文字在下方填寫。

In [ ]:
import numpy as np
import pandas as pd

# 先讀入（僅用於 A.0 探索，A.1 會正式 load）
_df = pd.read_csv("./data/messy_stroop_homework.csv")

print("=== dtypes ===")
print(_df.dtypes)
print("\n=== describe(include='all') ===")
display(_df.describe(include="all"))
print("\n=== isnull().sum() ===")
print(_df.isnull().sum())
print("\n=== value_counts for object columns ===")
for col in _df.select_dtypes("object"):
    print(f"\n--- {col} ---")
    print(_df[col].value_counts(dropna=False))

**來源 3 的觀察與比對**

1. **rt_ms dtype 與生成器不一致**：生成器第 43 行產出 float64，但 `df.dtypes` 顯示 `object`（因第 59 行 `astype(object)` 注入）。觀察到 `value_counts` 中出現 `"missing"`、`"--"` 字串，確認兩種字串 sentinel。

2. **rt_ms 數值範圍異常**：`describe()` 的 `min`/`max`（若能轉型）應顯示 -1 與 9999，均超出任何合理 RT 範圍（文獻或生成器均不支持 <0 或 >3000 的真實 RT）。

3. **condition 變體數**：`value_counts` 顯示 6 個不同字串（"congruent"、"congruent "、"incongruent"、"Incongruent"、"con"、"incong."），而來源 1 告訴我們邏輯上只應有 2 個 level。

4. **accuracy dtype 與編碼不一致**：`dtypes` 顯示 `object`（應為 int），且 `value_counts` 同時出現 0、1、`"True"`、`"False"`。

5. **age sentinel 三種**：`value_counts(dropna=False)` 可見 -1、888 與 `NaN` 同時存在；生成器第 45 行確認這是設計行為。

6. **「只從資料看不出來但生成器有告訴你」的規則**：資料中的 9999 可能被誤判為「極慢的真實 RT」（因為沒有文字說明），但生成器第 80–81 行明確標注這是 timeout sentinel。若僅靠 `describe()` 或文獻範圍（< 2500 ms）可以推斷它異常，但「它代表什麼」只有生成器能確認。

---
### 最終產出：Schema 表

整合三個來源後的欄位規則表：

| 欄位 | 預期型別 | 合理範圍 | 已知 sentinel | 依據 |
|------|----------|----------|---------------|------|
| `trial_id` | int | 1–240，每列唯一 | 無（但有 3 列重複） | 生成器第 38、108–110 行 |
| `subject_id` | int | {101, 102, 103, 104, 105, 106}，6 個值 | 無 | 生成器第 39 行 |
| `condition` | category | 2 levels：congruent / incongruent | 無（但有 6 個不一致字面值需標準化） | 生成器第 40–42、89–91 行；來源 3 value_counts |
| `rt_ms` | float | **200–3000 ms** | `"missing"`、`"--"`（字串）；`-1`、`9999`（數值） | • Ménétré & Laganaro (2023) Sec. 2.4：RT < 200 ms 視為猜測，採此絕對下限（生理原則，可跨樣本移植）。<br>• 文獻 SD-based 上限（1074.5 ms）依賴該樣本 mean=682 ms，與本資料 mean≈520 ms（生成器第 43 行）不相容，**不直接採用**；改採 demo 保守上限 3000 ms 保留試次。<br>• SD trimming 精神移至 `analyse(outlier_sd=3.0)` 對本資料實作。<br>• 取捨：下限比 demo（150 ms）嚴格，有文獻絕對閾值支持；上限保守以防小樣本（~240 trials）過度損失。|
| `accuracy` | int | 0 或 1 | `"True"`、`"False"`（字串） | 生成器第 43、96–101 行；來源 3 value_counts |
| `age` | float | 22–74（偶數，步距 4） | `-1`、`888`、`NaN` | 生成器第 44–45 行；來源 3 value_counts |

---
## A.1 Load

純 I/O，不做任何 transformation。

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_PATH = "./data/messy_stroop_homework.csv"

df_raw = pd.read_csv(DATA_PATH)
print(f"Loaded: {DATA_PATH}")
print(f"Shape: {df_raw.shape}")

---
## A.2 Inspect

第一眼檢視：shape、dtype、前幾列。

In [ ]:
print("Shape:", df_raw.shape)
print("\ndtypes:")
print(df_raw.dtypes)
print("\nFirst 5 rows:")
df_raw.head(5)

**第一眼觀察（2–4 行）**

資料有 243 列（比預期的 240 多 3，疑有重複列）和 6 個欄位。`rt_ms` 與 `accuracy` 的 dtype 為 `object` 而非數值，表示欄中混有字串；`condition` 有多個外觀不同的值；`age` 欄位看到 -1 與 888 等明顯異常數值。整體而言，**四個欄位有 dtype 或 encoding 問題**，需在 A.3 深入確認。

---
## A.3 Describe — 健康檢查

逐一執行標準健康檢查指令，並在每段程式碼後解讀發現。

In [ ]:
df_raw.info()

In [ ]:
df_raw.describe(include="all")

In [ ]:
print("缺值統計：")
print(df_raw.isnull().sum())

In [ ]:
for col in df_raw.select_dtypes("object"):
    print(f"\n=== {col} ===")
    print(df_raw[col].value_counts(dropna=False))

In [ ]:
# 視覺化：rt_ms 分佈（轉數值後）+ 各欄缺值數
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

rt_numeric = pd.to_numeric(df_raw["rt_ms"], errors="coerce")
axes[0].hist(rt_numeric.dropna(), bins=35, edgecolor="black", color="steelblue")
axes[0].axvline(200,  color="red",   linestyle="--", label="lower bound 200 ms (Ménétré & Laganaro, 2023)")
axes[0].axvline(3000, color="orange", linestyle="--", label="upper bound 3000 ms (demo)")
axes[0].set_title("rt_ms distribution (numeric only, non-numeric excluded)")
axes[0].set_xlabel("RT (ms)")
axes[0].set_ylabel("Count")
axes[0].legend(fontsize=8)

missing = df_raw.isnull().sum()
axes[1].bar(missing.index, missing.values, color="salmon", edgecolor="black")
axes[1].set_title("Missing values per column (NaN only)")
axes[1].set_ylabel("Count")

plt.tight_layout()
plt.show()
print("注意：rt_ms 的非數值字串 sentinel 不計入 NaN，需另行 value_counts 確認。")

**A.3 資料品質問題清單**

| # | 問題描述 | 觀察到的指令 | 具體數字 | 可能原因 |
|---|----------|-------------|---------|----------|
| 1 | `rt_ms` 為 object dtype，含字串 sentinel | `df.dtypes` + `value_counts` | `"missing"` 8 筆、`"--"` 5 筆 | 生成器注入後 `astype(object)` |
| 2 | `rt_ms` 含數值 sentinel（兩個方向） | `value_counts`（轉型後 describe min/max） | -1（4 筆）、9999（4 筆） | 分別代表無反應 / timeout |
| 3 | `condition` 有 6 個變體（應為 2） | `value_counts` | 見 value_counts 輸出 | 尾端空白、大小寫、縮寫 |
| 4 | `accuracy` 為 object dtype，含字串 bool | `df.dtypes` + `value_counts` | `"True"` 5 筆、`"False"` 5 筆 | 不同受試者用不同編碼 |
| 5 | `age` 含三種 missing 表示 | `value_counts(dropna=False)` | -1（?筆）、888（?筆）、NaN（?筆） | 同欄多種 sentinel |
| 6 | `trial_id` 有重複（總列數 243 ≠ 240） | `df.shape` + `duplicated()` | 3 列完整重複 | 資料合併失誤模擬 |

> *（請在執行上方程式碼後，將「?」填入實際觀察數字。）*

---
## A.4 Fix — Observation-driven cleaning

`clean()` 是 pure function：不修改輸入 df，每步印出 row 數變化，每條規則連結回 A.0 schema 表。

In [ ]:
def clean(df: pd.DataFrame) -> pd.DataFrame:
    """
    Observation-driven cleaning for messy_stroop_homework.csv.
    每個步驟格式：
      觀察 → 對應 A.3 哪一條
      動作 → 用了哪個 pandas API
      代價 → 這個動作犧牲了什麼
    """
    df = df.copy()

    # ------------------------------------------------------------------
    # Step 1: 去除完整重複列
    # 觀察：A.3 問題 6 — df.shape 顯示 243 列，trial_id 有 3 個重複
    # 動作：drop_duplicates() 保留第一次出現
    # 代價：若重複是真實的不同 trial 只是 trial_id 碰撞，會漏掉 trial；
    #        但生成器確認（第 108–110 行）這是完整列複製，安全刪除
    # ------------------------------------------------------------------
    before = len(df)
    df = df.drop_duplicates()
    print(f"Step 1 drop_duplicates:         {before} → {len(df)}  (lost {before - len(df)} rows)")

    # ------------------------------------------------------------------
    # Step 2: 標準化 condition → 2 個 level
    # 觀察：A.3 問題 3 — value_counts 顯示 6 個字面變體
    # 動作：strip + lower 處理空白/大小寫；replace 處理縮寫
    # 代價：將 "con" / "incong." 對應到單一 level 需假設縮寫語意；
    #        由來源 1 第 90–91 行已確認語意，無歧義
    # ------------------------------------------------------------------
    before = len(df)
    df["condition"] = df["condition"].str.strip().str.lower()
    abbrev_map = {"con": "congruent", "incong.": "incongruent"}
    df["condition"] = df["condition"].replace(abbrev_map)
    df["condition"] = df["condition"].astype("category")
    print(f"Step 2 standardize condition:   {before} → {len(df)}  (lost 0 rows)")
    print(f"       condition levels: {sorted(df['condition'].cat.categories.tolist())}")

    # ------------------------------------------------------------------
    # Step 3: 修復 rt_ms dtype — 字串 sentinel → NaN，再轉 float
    # 觀察：A.3 問題 1 — object dtype，"missing" 8 筆、"--" 5 筆
    # 動作：pd.to_numeric(errors="coerce") 將無法解析的字串轉為 NaN
    # 代價：字串 sentinel 的語意資訊統一為 NaN 丟失
    # ------------------------------------------------------------------
    before = len(df)
    df["rt_ms"] = pd.to_numeric(df["rt_ms"], errors="coerce")
    n_nan = df["rt_ms"].isnull().sum()
    print(f"Step 3 to_numeric(rt_ms):       {before} → {len(df)}  (lost 0 rows; {n_nan} NaN after coerce)")

    # ------------------------------------------------------------------
    # Step 4: 替換 rt_ms 數值 sentinel → NaN
    # 觀察：A.3 問題 2 — -1（4 筆）、9999（4 筆）出現在 value_counts
    # 動作：replace()
    # 代價：來源 1（生成器第 74–81 行）+ 來源 2（下限 200 ms）雙重確認均為 sentinel；
    #        -1 物理上不可能，9999 遠超文獻合理範圍，安全替換
    # ------------------------------------------------------------------
    before = len(df)
    df["rt_ms"] = df["rt_ms"].replace({-1: np.nan, 9999: np.nan})
    print(f"Step 4 replace rt_ms sentinels: {before} → {len(df)}  (lost 0 rows)")

    # ------------------------------------------------------------------
    # Step 5: 過濾 rt_ms 超出合理範圍的 row
    # 觀察：A.3 histogram — 非 sentinel 值均在合理區間內
    # 動作：先 dropna 再 between；範圍依 A.0 schema 表：200–3000 ms
    #   下限 200 ms — Ménétré & Laganaro (2023) Sec. 2.4 絕對閾值
    #   上限 3000 ms — 課堂 demo 保守上限（文獻 SD-based 上限不可跨樣本移植）
    # 代價：< 200 ms 的真實極速反應（幾乎不存在）會被刪除；
    #        > 3000 ms 的極端慢速 trial 被視為離群，損失部分數據
    # ------------------------------------------------------------------
    before = len(df)
    df = df.dropna(subset=["rt_ms"])
    print(f"Step 5a dropna(rt_ms):          {before} → {len(df)}  (lost {before - len(df)} rows)")
    before = len(df)
    df = df[df["rt_ms"].between(200, 3000)]
    print(f"Step 5b rt_ms between(200,3000): {before} → {len(df)}  (lost {before - len(df)} rows)")

    # ------------------------------------------------------------------
    # Step 6: 修復 accuracy — 字串 "True"/"False" → 1/0，轉 int
    # 觀察：A.3 問題 4 — object dtype，"True" 5 筆、"False" 5 筆
    # 動作：replace() 映射字串 → 數值，再 pd.to_numeric + astype(Int64)
    # 代價：由來源 1 第 99–100 行確認 "True"=1、"False"=0，語意一致，安全映射
    # ------------------------------------------------------------------
    before = len(df)
    df["accuracy"] = df["accuracy"].replace({"True": 1, "False": 0})
    df["accuracy"] = pd.to_numeric(df["accuracy"], errors="coerce").astype("Int64")
    print(f"Step 6 fix accuracy dtype:      {before} → {len(df)}  (lost 0 rows)")

    # ------------------------------------------------------------------
    # Step 7: 替換 age sentinel → NaN（不 drop，age 不影響 RT 主分析）
    # 觀察：A.3 問題 5 — value_counts 顯示 -1 與 888 兩種數值 sentinel
    # 動作：replace()
    # 代價：-1 與 888 在人類年齡上物理不可能，安全替換；NaN 列保留
    # ------------------------------------------------------------------
    before = len(df)
    df["age"] = df["age"].replace({-1: np.nan, 888: np.nan})
    print(f"Step 7 replace age sentinels:   {before} → {len(df)}  (lost 0 rows)")

    return df


df_clean = clean(df_raw)
print(f"\n最終：{len(df_raw)} → {len(df_clean)} rows")

---
## A.5 Re-describe — 驗證 fix

對清理後的 df 再跑一次 describe，比對 A.3 的問題是否消失。

In [ ]:
print("=== dtypes（清理後）===")
print(df_clean.dtypes)
print("\n=== describe（清理後）===")
display(df_clean.describe(include="all"))
print("\n=== isnull().sum()（清理後）===")
print(df_clean.isnull().sum())
print("\n=== condition value_counts ===")
print(df_clean["condition"].value_counts(dropna=False))
print("\n=== accuracy value_counts ===")
print(df_clean["accuracy"].value_counts(dropna=False))
print("\n=== trial_id duplicates ===")
print("重複 trial_id 數：", df_clean["trial_id"].duplicated().sum())

**A.5 驗證結果（與 A.3 對比）**

> *（執行上方程式碼後，在此填入 1–2 段觀察文字，例如：）*

相比 A.3，清理後：

- `rt_ms` dtype 由 `object` 改為 `float64`，`min` 不再出現 -1，`max` 不再出現 9999。
- `condition` 由 6 個變體縮減為 2 個（congruent / incongruent），dtype 為 `category`。
- `accuracy` dtype 由 `object` 改為 `Int64`，`value_counts` 只剩 0 / 1。
- `age` 的 -1 與 888 消失，改為 NaN，不影響行數（未 drop age 缺值列）。
- 總列數由 243 降至？列（去重 3、rt_ms NaN 及超範圍共？列）。

---
## A.6 Analyse — 回答研究問題

計算兩個條件的 mean RT 與 SD，以及 Stroop effect。

`outlier_sd` 參數放在 `analyse()` 而非 `clean()` 的原因：outlier 門檻是**分析層的研究決策**（不同研究問題可能採不同門檻），而非資料本身的「錯誤」。`clean()` 只修正客觀的 dirty pattern（dtype 錯誤、sentinel、重複列），不應混入會改變研究結論的主觀判斷；`analyse()` 可用不同 `outlier_sd` 做 sensitivity analysis，保留分析彈性，符合本週課程強調的 cleaning / analysis 邊界紀律。

In [ ]:
def analyse(df: pd.DataFrame, *, outlier_sd: float = 3.0) -> dict:
    """
    計算 Stroop effect。

    Parameters
    ----------
    df : 清理後的 DataFrame（輸出自 clean()）
    outlier_sd : 以 mean ± N * SD 過濾 rt_ms outlier（分析層決策）

    Returns
    -------
    dict with keys: mean_cong, sd_cong, mean_incong, sd_incong,
                    stroop_effect, n_cong, n_incong
    """
    df = df.copy()

    # Outlier 過濾：分析層決策，不放在 clean()
    mu = df["rt_ms"].mean()
    sigma = df["rt_ms"].std()
    before = len(df)
    df = df[df["rt_ms"].between(mu - outlier_sd * sigma,
                                 mu + outlier_sd * sigma)]
    print(f"Outlier filter (±{outlier_sd} SD): {before} → {len(df)} rows")

    grp = df.groupby("condition")["rt_ms"]
    stats = grp.agg(["mean", "std", "count"]).rename(
        columns={"mean": "Mean RT", "std": "SD", "count": "N"}
    )
    print("\n=== Condition statistics ===")
    display(stats)

    mean_cong   = stats.loc["congruent",   "Mean RT"]
    mean_incong = stats.loc["incongruent", "Mean RT"]
    stroop_effect = mean_incong - mean_cong

    print(f"\nStroop effect = {mean_incong:.1f} − {mean_cong:.1f} = {stroop_effect:.1f} ms")
    print(f"（生成器注入 +80 ms；期望回收值約 70–90 ms，視 outlier 過濾量而定）")

    return {
        "mean_cong":     mean_cong,
        "sd_cong":       stats.loc["congruent",   "SD"],
        "n_cong":        int(stats.loc["congruent",   "N"]),
        "mean_incong":   mean_incong,
        "sd_incong":     stats.loc["incongruent", "SD"],
        "n_incong":      int(stats.loc["incongruent", "N"]),
        "stroop_effect": stroop_effect,
    }


results = analyse(df_clean, outlier_sd=3.0)

In [ ]:
# 視覺化：condition mean RT bar chart with SD error bar
conditions = ["congruent", "incongruent"]
means = [results["mean_cong"], results["mean_incong"]]
sds   = [results["sd_cong"],   results["sd_incong"]]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(conditions, means, yerr=sds, capsize=6,
              color=["steelblue", "salmon"], edgecolor="black", alpha=0.85)
ax.set_ylabel("Mean RT (ms)")
ax.set_title(f"Stroop Effect = {results['stroop_effect']:.1f} ms\n(mean ± 1 SD, outlier_sd=3.0)")
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width() / 2, m + 5, f"{m:.1f}",
            ha="center", va="bottom", fontsize=10)
plt.tight_layout()
plt.show()